# Code extracted from `interprpject.pdf`

This notebook reproduces the Python snippets from the PDF (macro Logistic regression, meso PCA via Wind, micro structure percentiles). Paths point to the author’s machine; update CSV/Wind usage before running.

Minor fixes applied where the PDF broke lines or omitted a variable: merged `data = w.edb(...)`, `common_index = ...`, and multi-line plot labels; added `roc_auc = roc_auc_score(...)`; `df_rolling.dropna()` is assigned back to `df_rolling`.

## 1. Macro: Logistic regression

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, roc_auc_score, roc_curve
import matplotlib.pyplot as plt

df = pd.read_csv("/Users/haolinliu/Downloads/China_M1_YoY .csv")
df['date'] = pd.to_datetime(df['Name'], errors='coerce')
df = df.dropna(subset=['date'])
df.set_index('date', inplace=True)
df = df.bfill()
df = df.dropna(axis=1, how='all')
df = df.dropna(how='any')
results = {}
for column in df.columns:
    if column == 'Name':
        continue
    results[column] = pd.DataFrame()
    results[column][column] = df[column]
    results[column]['lag1'] = results[column][column].shift(1)
    results[column]['DIF'] = results[column][column] - results[column]['lag1']
    results[column]['EMA12'] = results[column][column].ewm(span=12, adjust=False).mean()
    results[column]['EMA26'] = results[column][column].ewm(span=26, adjust=False).mean()
    results[column]['DIF_EMA'] = results[column]['EMA12'] - results[column]['EMA26']
processed_df = pd.concat(results.values(), axis=1)
processed_df = processed_df.dropna()
data = pd.read_csv("/Users/haolinliu/Desktop/112233.csv", parse_dates=['交易日期'])  # PDF used 交易⽇期; match your CSV header
data = data.dropna(axis=1, how='all')
data.set_index('交易日期', inplace=True)
start_date = '2017-01-04'
end_date = '2023-03-27'
train_data = data.loc[start_date:end_date]
train_data['future_60d_return'] = train_data['收盘价'].shift(-60) / train_data['收盘价'] - 1
train_data['macro_environment'] = (train_data['future_60d_return'] > 0).astype(int)
train_data = train_data.dropna(subset=['future_60d_return'])
macro_environment_counts = train_data['macro_environment'].value_counts()
train_data = train_data.iloc[::-1]
train_data = train_data.join(processed_df, how='inner')
X = train_data[processed_df.columns]
y = train_data['macro_environment']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
model = LogisticRegression()
model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)
y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy}")
roc_auc = roc_auc_score(y_test, y_pred_proba)
fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label='Logistic Regression (AUC = {:.2f})'.format(roc_auc))
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.show()


## 2. Meso: PCA on Wind macro series (`w.edb`)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
from WindPy import w

w.start()
data = w.edb(
    "M0000548,S0027908,S0027703,S0027379,M5207101,M0000557,M0000559,M5567905",
    "2001-12-31",
    "2023-12-31",
    "Fill=Previous",
)
data.Fields = list(range(1, 9))
df = pd.DataFrame(data.Data, index=data.Fields, columns=data.Times)
df = df.T
df = df.dropna()
scaler = StandardScaler()
scaled_data = scaler.fit_transform(df)
pca = PCA(n_components=1)
pca_data = pca.fit_transform(scaled_data)
pca_df = pd.DataFrame(data=pca_data, columns=['Principal Component'], index=df.index)
plt.figure(figsize=(10, 6))
plt.plot(pca_df.index, pca_df['Principal Component'])
plt.title('PCA Result')
plt.xlabel('Date')
plt.ylabel('Principal Component')
plt.grid()
plt.show()


## 3. Micro: volatility / PE / turnover / ERP percentiles (Wind)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from WindPy import w

w.start()

# 获取股票波动率数据
data = w.wsd(
    "000905.SH,000985.CSI,000852.SH,000300.SH",
    "volatilityratio",
    "2019-06-25",
    "2024-07-25",
    "CalculationTime=60;AnnualCoefficient=252;",
)
df = pd.DataFrame(data.Data, index=data.Codes, columns=data.Times).T
# 计算滚动平均值
df_rolling = df.rolling(window=20).mean()
df_rolling = df_rolling.dropna()


# 定义计算分位数的函数
def calculate_percentiles(df):
    df_percentiles = df.apply(lambda x: x.rank(pct=True))
    return df_percentiles


df_percentiles = calculate_percentiles(df_rolling)
# 绘制波动率分位数图
plt.figure(figsize=(12, 6))
plt.plot(df_percentiles.index, df_percentiles, label='Mean Percentile')
plt.title('Mean Percentiles of Rolling 20-week Volatility Over Last 5 Years')
plt.xlabel('Date')
plt.ylabel('Mean Percentile')
plt.legend(df_percentiles.columns)
plt.grid()
plt.show()

# 获取市盈率数据
pe_data = w.wsd(
    "000905.SH,000985.CSI,000852.SH,000300.SH",
    "pe",
    "2019-06-25",
    "2024-07-25",
    "ruleType=10;",
)
pe_df = pd.DataFrame(pe_data.Data, index=pe_data.Codes, columns=pe_data.Times).T
pe_df = pe_df.rolling(window=20).mean()
# 计算市盈率分位数
pe_df_percentiles = calculate_percentiles(pe_df)
plt.figure(figsize=(12, 6))
plt.plot(pe_df_percentiles.index, pe_df_percentiles, label='Mean Percentile')
plt.title('Mean Percentiles of Rolling 20-week Price-earnings ratio Over Last 5 Years')
plt.xlabel('Date')
plt.ylabel('Mean Percentile')
plt.legend(pe_df_percentiles.columns)
plt.grid()
plt.show()

# 获取换手率数据
turn_data = w.wsd(
    "000905.SH,000985.CSI,000852.SH,000300.SH",
    "turn",
    "2019-06-25",
    "2024-07-25",
    "",
)
turn_df = pd.DataFrame(turn_data.Data, index=turn_data.Codes, columns=turn_data.Times).T
turn_df_percentiles = calculate_percentiles(turn_df)
# 计算换手率分位数
plt.figure(figsize=(12, 6))
plt.plot(turn_df_percentiles.index, turn_df_percentiles, label='Mean Percentile')
plt.title('Mean Percentiles of Rolling 20-week Turn ratio Over Last 5 Years')
plt.xlabel('Date')
plt.ylabel('Mean Percentile')
plt.legend(turn_df_percentiles.columns)
plt.grid()
plt.show()

# 计算ERP分位数
data = w.wsd(
    "000905.SH,000985.CSI,000300.SH,000852.SH",
    "pe",
    "2019-06-26",
    "2024-07-26",
    "ruleType=10;",
)
df = pd.DataFrame(data.Data, index=data.Codes, columns=data.Times).T
df = 1 / (df / 100)
data_b = w.edb("S0059749", "2019-06-26", "2024-07-26", "Fill=Previous")
df_b = pd.DataFrame(data_b.Data, index=data_b.Codes, columns=data_b.Times).T
df_b = df_b.squeeze()
result = df.sub(df_b, axis=0)
result = -result
result = calculate_percentiles(result)
plt.figure(figsize=(12, 6))
plt.plot(result.index, result, label='Mean Percentile')
plt.title('Mean Percentiles of ERP Over Last 5 Years')
plt.xlabel('Date')
plt.ylabel('Mean Percentile')
plt.legend(result.columns)
plt.grid()
plt.show()

# 对齐所有分位数的DataFrame并计算它们的和
common_index = (
    df_percentiles.index.intersection(pe_df_percentiles.index)
    .intersection(turn_df_percentiles.index)
    .intersection(result.index)
)
df_percentiles_aligned = df_percentiles.loc[common_index]
pe_df_percentiles_aligned = pe_df_percentiles.loc[common_index]
turn_df_percentiles_aligned = turn_df_percentiles.loc[common_index]
result_aligned = result.loc[common_index]
# 将所有分位数求和并计算平均值
summed_percentiles = df_percentiles_aligned.add(pe_df_percentiles_aligned, fill_value=0)
summed_percentiles = summed_percentiles.add(turn_df_percentiles_aligned, fill_value=0)
summed_percentiles = summed_percentiles.add(result_aligned, fill_value=0)
summed_percentiles = summed_percentiles / 4

# 绘制所有单独分位数和它们的总和
plt.figure(figsize=(12, 8))
plt.plot(
    df_percentiles_aligned.index,
    df_percentiles_aligned.mean(axis=1),
    label='Volatility Percentile',
)
plt.plot(
    pe_df_percentiles_aligned.index,
    pe_df_percentiles_aligned.mean(axis=1),
    label='PE Ratio Percentile',
)
plt.plot(
    turn_df_percentiles_aligned.index,
    turn_df_percentiles_aligned.mean(axis=1),
    label='Turnover Percentile',
)
plt.plot(result_aligned.index, result_aligned.mean(axis=1), label='ERP Percentile')
plt.plot(
    summed_percentiles.index,
    summed_percentiles.mean(axis=1),
    label='Sum of Percentiles',
    color='black',
    linewidth=2,
)
plt.title('Individual and Summed Percentiles of Different Market Indicators')
plt.xlabel('Date')
plt.ylabel('Percentile')
plt.legend()
plt.grid()
plt.show()
